# 📝 Gerador de Legendas para Vídeo

Gera arquivos de legenda **SRT** a partir de um arquivo de áudio ou vídeo.  
Usa **faster-whisper** (OpenAI Whisper otimizado) — 100% gratuito, roda localmente, sem API key.

## O que este notebook faz:
1. Transcreve o áudio do vídeo ou arquivo MP3
2. Gera arquivo `.srt` com timing preciso para cada frase
3. Gera arquivo `.vtt` (alternativo para alguns editores)
4. Mostra instruções de como importar no **Camtasia Studio 8**

> **Dica**: Se você usou o notebook `07_tts_narration_generator.ipynb`, já existe um `.srt` sincronizado automaticamente na pasta `outputs/audio/`. Use este notebook para gerar legendas a partir de um vídeo gravado ou narração própria.

## 1. Instalar dependências

In [ ]:
# faster-whisper: versão otimizada do Whisper da OpenAI (4x mais rápido, menos memória)
!pip install faster-whisper

# Para extrair áudio de vídeo MP4 (necessita ffmpeg no sistema)
# Windows: choco install ffmpeg  ou baixe em https://ffmpeg.org/download.html
print('Dependências instaladas!')

## 2. Configuração

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES — edite aqui
# ============================================================

# Caminho do arquivo de entrada (MP3, MP4, WAV, etc.)
# Exemplos:
#   '../outputs/audio/narracao_imovel_feminina.mp3'  ← áudio gerado pelo notebook 07
#   'C:/Videos/meu_video.mp4'                        ← vídeo completo
ARQUIVO_ENTRADA = '../outputs/audio/narracao_imovel_feminina.mp3'

# Modelo Whisper (tamanho x qualidade x velocidade):
#   'tiny'   → mais rápido, qualidade menor  (bom para testes)
#   'base'   → rápido, qualidade boa          ← recomendado para PT-BR simples
#   'small'  → mais lento, qualidade melhor   ← melhor custo-benefício
#   'medium' → lento, qualidade alta          ← se tiver GPU ou paciência
#   'large-v3' → mais lento, máxima qualidade
MODELO = 'small'

# Idioma (use 'pt' para Português)
IDIOMA = 'pt'

# Máximo de caracteres por linha de legenda
MAX_CHARS_POR_LINHA = 60

# Pasta de saída para os arquivos de legenda
PASTA_SAIDA = Path('../outputs/legendas')

# ============================================================

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
arquivo_entrada = Path(ARQUIVO_ENTRADA)
nome_base = arquivo_entrada.stem

print(f'Arquivo de entrada: {arquivo_entrada}')
print(f'Modelo: {MODELO} | Idioma: {IDIOMA}')
print(f'Saída em: {PASTA_SAIDA}')

if not arquivo_entrada.exists():
    print('\n⚠️  Arquivo de entrada não encontrado!')
    print('   Verifique o caminho em ARQUIVO_ENTRADA')
else:
    import os
    kb = os.path.getsize(arquivo_entrada) / 1024
    print(f'   Tamanho: {kb:.1f} KB ✅')

## 3. Transcrever o áudio

In [ ]:
from faster_whisper import WhisperModel

print(f'Carregando modelo "{MODELO}"...')
print('(primeira execução faz download do modelo — pode demorar alguns minutos)\n')

# device='cpu' funciona em qualquer máquina
# device='cuda' usa GPU NVIDIA se disponível (muito mais rápido)
model = WhisperModel(MODELO, device='cpu', compute_type='int8')

print('Transcrevendo... aguarde.')
segments, info = model.transcribe(
    str(arquivo_entrada),
    language=IDIOMA,
    beam_size=5,
    vad_filter=True,          # filtra silêncio automaticamente
    vad_parameters=dict(min_silence_duration_ms=500),
    word_timestamps=True,      # timing por palavra (maior precisão)
)

# Materializa os segmentos
segmentos = list(segments)

print(f'\n✅ Transcrição concluída!')
print(f'   Idioma detectado: {info.language} (confiança: {info.language_probability:.0%})')
print(f'   Duração: {info.duration:.1f}s | Segmentos: {len(segmentos)}')
print('\n--- Prévia da transcrição ---')
for seg in segmentos[:5]:
    print(f'  [{seg.start:.1f}s → {seg.end:.1f}s] {seg.text.strip()}')

## 4. Gerar arquivo SRT (formato padrão de legendas)

In [ ]:
def segundos_para_srt(segundos: float) -> str:
    """Converte segundos para formato SRT: HH:MM:SS,mmm"""
    h = int(segundos // 3600)
    m = int((segundos % 3600) // 60)
    s = int(segundos % 60)
    ms = int((segundos % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def quebrar_linha(texto: str, max_chars: int) -> str:
    """Quebra texto em duas linhas se muito longo."""
    if len(texto) <= max_chars:
        return texto
    palavras = texto.split()
    meio = len(palavras) // 2
    linha1 = ' '.join(palavras[:meio])
    linha2 = ' '.join(palavras[meio:])
    return f'{linha1}\n{linha2}'

def gerar_srt(segmentos, max_chars=60) -> str:
    linhas = []
    for i, seg in enumerate(segmentos, start=1):
        inicio = segundos_para_srt(seg.start)
        fim    = segundos_para_srt(seg.end)
        texto  = quebrar_linha(seg.text.strip(), max_chars)
        linhas.append(f'{i}\n{inicio} --> {fim}\n{texto}\n')
    return '\n'.join(linhas)

conteudo_srt = gerar_srt(segmentos, MAX_CHARS_POR_LINHA)
arquivo_srt = PASTA_SAIDA / f'{nome_base}.srt'

with open(arquivo_srt, 'w', encoding='utf-8') as f:
    f.write(conteudo_srt)

print(f'✅ SRT gerado: {arquivo_srt}')
print(f'\n--- Prévia do SRT ---')
print(conteudo_srt[:800])

## 5. Gerar arquivo VTT (alternativo — WebVTT)

In [ ]:
def segundos_para_vtt(segundos: float) -> str:
    """Converte segundos para formato VTT: HH:MM:SS.mmm"""
    h = int(segundos // 3600)
    m = int((segundos % 3600) // 60)
    s = int(segundos % 60)
    ms = int((segundos % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d}.{ms:03d}'

def gerar_vtt(segmentos, max_chars=60) -> str:
    linhas = ['WEBVTT\n']
    for i, seg in enumerate(segmentos, start=1):
        inicio = segundos_para_vtt(seg.start)
        fim    = segundos_para_vtt(seg.end)
        texto  = quebrar_linha(seg.text.strip(), max_chars)
        linhas.append(f'{i}\n{inicio} --> {fim}\n{texto}\n')
    return '\n'.join(linhas)

conteudo_vtt = gerar_vtt(segmentos, MAX_CHARS_POR_LINHA)
arquivo_vtt = PASTA_SAIDA / f'{nome_base}.vtt'

with open(arquivo_vtt, 'w', encoding='utf-8') as f:
    f.write(conteudo_vtt)

print(f'✅ VTT gerado: {arquivo_vtt}')

## 6. Exibir legenda completa e transcrição do texto

In [ ]:
print('=== TRANSCRIÇÃO COMPLETA ===')
texto_completo = ' '.join(seg.text.strip() for seg in segmentos)
print(texto_completo)

# Salva também como TXT para fácil revisão
arquivo_txt = PASTA_SAIDA / f'{nome_base}_transcricao.txt'
with open(arquivo_txt, 'w', encoding='utf-8') as f:
    f.write(texto_completo)

print(f'\n✅ Transcrição TXT salva: {arquivo_txt}')
print(f'\nArquivos gerados:')
print(f'  📄 {arquivo_srt.name}  ← use no Camtasia')
print(f'  📄 {arquivo_vtt.name}  ← alternativo')
print(f'  📄 {arquivo_txt.name}  ← texto puro')

---
## 📺 Como adicionar legendas no Camtasia Studio 8

### Método 1 — Importar arquivo SRT (recomendado)

O **Camtasia Studio 8** suporta importação de legendas no formato `.srt`:

1. **Abra seu projeto** no Camtasia Studio 8
2. No menu superior, vá em **`Edit`** → **`Captions`**
3. Na janela de Captions que abrir, clique em **`Import Captions`**
4. Selecione o arquivo **`.srt`** gerado neste notebook
5. As legendas aparecerão automaticamente na timeline sincronizadas com o vídeo
6. Você pode ajustar posição, fonte e cor clicando em cada legenda

> **Se não aparecer a opção de Import**: No Camtasia 8, vá em `Tools` → `Captions`

---

### Método 2 — Digitar legendas manualmente

1. Vá em **`Edit`** → **`Captions`**
2. Posicione o cursor na timeline onde quer a legenda
3. Clique em **`Add Caption`** e digite o texto
4. Arraste as bordas da legenda na timeline para ajustar a duração

---

### Método 3 — "Queimar" legenda no vídeo (burnt-in subtitles)

Se quiser a legenda permanente (visível em qualquer player), use o **FFmpeg** no terminal:

```bash
# Substitua os caminhos abaixo pelos seus arquivos
ffmpeg -i meu_video.mp4 -vf subtitles=narracao_imovel_feminina.srt video_com_legenda.mp4
```

---

### Dicas para Camtasia Studio 8 + Áudio gerado

| Passo | Ação no Camtasia |
|-------|------------------|
| 1 | `File > Import Media` — importe o MP3 da narração |
| 2 | Arraste o MP3 para a **Track 2** da timeline (abaixo do vídeo) |
| 3 | Ajuste o volume da narração: clique direito na faixa → `Audio` |
| 4 | Importe o SRT via `Edit > Captions > Import Captions` |
| 5 | Exporte o vídeo final: `File > Produce and Share` → MP4 |

---

### Arquivos gerados neste notebook:
```
outputs/legendas/
├── narracao_imovel_feminina.srt    ← importar no Camtasia
├── narracao_imovel_feminina.vtt    ← formato alternativo WebVTT
└── narracao_imovel_feminina_transcricao.txt  ← texto puro
```